# 课后练习解答（05.05_evaluation_and_dialogue_test）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** do_sample=False 且 temperature=0.8 时，生成方式为？
A. 贪心解码，temperature 不生效
B. 随机采样
C. top-k 采样
D. 温度采样

**解答：** A

**解析：** 贪心解码直接取最高概率 token，temperature 只在采样路径生效。


### 问题2（单选题）

**题目：** 输入长度 900、max_new_tokens=128、模型窗口 1024，会发生？
A. 总长度 1028 超出窗口，可能报错或截断
B. 一定正常
C. 自动忽略输入
D. 只生成 124 tokens

**解答：** A

**解析：** 900+128=1028 超过上下文窗口，需要截断输入或减小 max_new_tokens。


### 问题3（多选题）

**题目：** 降低回答重复的手段包括？
A. repetition_penalty
B. no_repeat_ngram_size
C. 合适的 temperature/top_p
D. 增大 max_new_tokens

**解答：** ABC

**解析：** 增大输出长度通常会增加重复风险，不是抑制手段。


### 问题4（多选题）

**题目：** 心理咨询场景对话评估维度包括？
A. 专业准确性
B. 共情表达
C. 建议可执行性
D. 安全性

**解答：** ABCD

**解析：** 专业、共情、可执行与安全共同决定真实可用性。


### 问题5（判断题）

**题目：** torch.no_grad() 下调用 generate 可避免保存反向图，降低显存。

**解答：** 对

**解析：** 推理不需要梯度，不构建反向图即可释放激活内存。


### 问题6（判断题）

**题目：** do_sample=False 时，temperature 仍会改变输出分布。

**解答：** 错

**解析：** 贪心解码不做分布采样，temperature 对结果无影响。


### 问题7（填空题）

**题目：** top_p=0.9 表示从累积概率达到 0.9 的最小候选集合中采样，称为 ____。

**解答：** 核采样（nucleus sampling）


### 问题8（填空题）

**题目：** generate 中控制新增 token 数量的是 ____，控制停止的是 ____。

**解答：** max_new_tokens；eos_token_id/停止条件


### 问题9（简答题）

**题目：** 为什么回答截断既要处理 <|end|> 也要处理 <|user|>？

**解答：** <|end|> 是正常结束标记，但模型可能误生成 <|user|> 模拟下一轮；若只按 <|end|> 截断，混入的 user 标记会污染回答展示和后续多轮拼接。


### 问题10（简答题）

**题目：** 如何设计可复现的对话评估流程？

**解答：** 固定随机种子、固定 prompt 集合与生成参数，逐条记录输入输出和停止原因；由多人按统一评分表独立打分，最后汇总一致性，确保结果可复现可审计。


### 问题11（代码设计题）

**题目：** 编写 chat_infer(model, tokenizer, prompt, max_new_tokens, do_sample, temperature, top_p)，返回清理后的回答。

**解答：** ```python
def chat_infer(model, tokenizer, prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_p=0.9):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    answer = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    for stop in ["<|end|>", "<|user|>"]:
        idx = answer.find(stop)
        if idx != -1:
            answer = answer[:idx]
    return answer.strip()
```


### 问题12（单选题）

**题目：** 模型持续输出同一句话，最直接有效的参数是？
A. repetition_penalty
B. max_new_tokens
C. lora_alpha
D. batch_size

**解答：** A

**解析：** repetition_penalty 会惩罚重复 token，直接抑制循环输出。


### 问题13（多选题）

**题目：** 安全性评估应覆盖？
A. 有害请求拒绝
B. 隐私边界
C. 诱导性 prompt
D. 回答长度

**解答：** ABC

**解析：** 回答长度不是安全维度，安全重点是内容边界与风险行为。


### 问题14（判断题）

**题目：** temperature 越高，回答质量一定越高。

**解答：** 错

**解析：** 过高的 temperature 会增加随机性与答非所问风险。


### 问题15（简答题）

**题目：** 为心理咨询机器人设计 4 维度评分表，并说明每个维度的评分标准。

**解答：** 1) 专业度：诊断边界、术语与建议是否准确；2) 共情度：是否理解情绪并给出支持性回应；3) 可执行性：建议是否具体、分步、可落地；4) 安全性：是否规避伤害性建议、明确危机转介。每项 1-5 分并附示例证据。
